# Word2Vec on the Linewise subject

**Intro to NLP: training word embeddings for a class-concept product**

**The subject.** *Linewise* is a class-concept product: a short daily practice where students read, review and debug flawed AI-generated code (an off-by-one here, a mutable default argument there), so they can use AI assistants without losing the ability to check what the assistants write. The course documents describe the target user, her diary entries (anxiety before interviews, pride after finding a bug), the brand, and a product spec.

**What Word2Vec does.** Word2Vec learns a short list of numbers (a *vector*) for each word. This notebook uses the **skip-gram** variant: the model takes a *center* word and tries to predict the words around it (its *context*, here up to 5 words on each side). To get good at that, it has to give words that appear in **similar contexts** **similar vectors**. That is the *distributional hypothesis*: "you shall know a word by the company it keeps."

Once trained, we can measure how close two words are (cosine similarity), find a word's nearest neighbors, try analogies, and project the 100-dimension vectors down to 2D and 3D to see the structure.

**Steps:** 1. build the corpus, 2. train, 3. explore the vectors, 4. 2D graphs, 5. 3D graphs, 6. measure cluster quality, 7. conclusions.

*This notebook expects to be run from the `word2vec/` folder (paths like `model/...` are relative to it).*

## 0. Setup

We import the corpus builder and the training settings from the two project scripts so the notebook uses exactly the same data and parameters as `train_word2vec.py`. (That script switches matplotlib to a file-only backend when imported, so we switch back to inline plots afterwards.)

In [ ]:
import sys, warnings
sys.dont_write_bytecode = True          # don't leave __pycache__ behind
warnings.filterwarnings("ignore")       # hide harmless library warnings

import random
import numpy as np
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio

from build_corpus import build, GROUPS, SEED, course_sentences, generated_sentences
from train_word2vec import PARAMS, COLORS, evaluate, topic_words, pretty

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
pio.renderers.default = "notebook_connected"
np.random.seed(SEED)
print("seed:", SEED)

## 1. Build the corpus

The corpus has two parts:

1. **Real course text**: the Linewise course document (`data/course_prompt_chain.md`: target user, diary entries, brand system, product spec), split into sentences and lower-cased. It is only about 8,000 words, which is too small to train stable vectors on its own, so it is repeated 3 times in the corpus.
2. **Generated sentences**: short sentences produced from templates about the same subject (bug types, code constructs, review actions, languages, AI tools, practice, feelings, career).

**An honest caveat:** the generated sentences are *synthetic*. Each template is written for one topic group (for example `"she felt {w} when she found the bug"` is filled only with positive feelings), so words from the same group are placed in the same contexts on purpose. That is exactly why the topic clusters later come out so clean: the model is recovering structure we built into the data, not discovering it in natural text. It is still a fair demonstration of *how* Word2Vec turns shared contexts into nearby vectors.

Multi-word terms are joined into single tokens, e.g. "off by one" becomes `off_by_one` and "c++" becomes `cpp`.

In [ ]:
corpus, n_course, n_generated = build()
n_tokens = sum(len(s) for s in corpus)
print(f"real course sentences : {n_course:>6,}  (each used 3 times)")
print(f"generated sentences   : {n_generated:>6,}")
print(f"total sentences       : {len(corpus):>6,}")
print(f"total tokens          : {n_tokens:>6,}")

In [ ]:
print("Examples from the real course text:")
course = course_sentences()
for s in [s for s in course if 6 <= len(s) <= 18][:5]:
    print("  ", " ".join(s))

print("\nExamples of generated (synthetic template) sentences:")
gen = generated_sentences(random.Random(SEED))
for i in random.Random(SEED).sample(range(len(gen)), 6):
    print("  ", " ".join(gen[i]))

The topic groups below are used for two things: the templates fill sentences from them, and the graphs color words by them.

In [ ]:
for group, words in GROUPS.items():
    print(f"{group:20s} ({len(words):2d}) {', '.join(words)}")

## 2. Train the model

The settings come from `train_word2vec.PARAMS`:

- `vector_size=100`: each word becomes 100 numbers.
- `window=5`: context is up to 5 words on each side of the center word.
- `min_count=3`: words seen fewer than 3 times are ignored.
- `sg=1`: skip-gram (predict context from the center word), which tends to work better than CBOW on small corpora.
- `negative=10`: for each real (center, context) pair, the model also sees 10 random "fake" pairs and learns to tell them apart (negative sampling).
- `epochs=40`: 40 passes over the corpus.
- `seed` and `workers=1`: fixed seed and a single thread so the numbers are reproducible.

Training takes a few seconds, so we retrain here to keep the notebook self-contained.

In [ ]:
print(PARAMS)
model = Word2Vec(sentences=corpus, **PARAMS)
wv = model.wv
print(f"trained: {len(wv):,} words in the vocabulary, {wv.vector_size} dimensions each")

The training script also saved a model to disk. Loading it is one line; because the seed and single worker are the same, it should match the model we just trained.

In [ ]:
saved = Word2Vec.load("model/linewise_word2vec.model")
print("saved model vocabulary:", len(saved.wv))
same = len(saved.wv) == len(wv) and np.allclose(saved.wv[wv.index_to_key], wv.vectors, atol=1e-5)
print("identical to the model trained above:", same)

## 3. Explore the vectors

### 3.1 What a word vector looks like
Each word is just a list of 100 numbers. No single number means anything by itself; meaning lives in how vectors relate to each other.

In [ ]:
v = wv["off_by_one"]
print("vocabulary size:", len(wv))
print("vector for 'off_by_one': shape", v.shape)
print("first 10 numbers:", np.round(v[:10], 3))

### 3.2 Nearest neighbors
`most_similar` ranks words by **cosine similarity** (1 = same direction, 0 = unrelated, negative = opposite). Words that appear in similar contexts should be neighbors.

In [ ]:
for w in ["off_by_one", "python", "copilot", "anxiety", "confidence", "interview"]:
    neighbors = wv.most_similar(w, topn=6)
    print(f"{w:12s} -> " + ", ".join(f"{n} ({s:.2f})" for n, s in neighbors))

Each probe's neighbors come from its own topic: bug types next to bug types, languages next to languages, negative feelings next to negative feelings.

### 3.3 Similarity of word pairs
Pairs from the same topic should score higher than pairs from different topics.

In [ ]:
pairs = [("anxiety", "shame"), ("anxiety", "confidence"), ("python", "java"),
         ("python", "anxiety"), ("off_by_one", "range"), ("deadlock", "thread"),
         ("copilot", "chatbot"), ("interview", "internship")]
for a, b in pairs:
    print(f"{a:>10s} ~ {b:<11s} {wv.similarity(a, b):6.3f}")

### 3.4 Analogies
"*a* is to *b* as *c* is to ?" is answered with vector arithmetic: find the word closest to **b − a + c**. These are the three analogies from `train_word2vec.evaluate`.

In [ ]:
words, labels = topic_words(wv)
vectors = np.array([wv[w] for w in words])
results = evaluate(wv, vectors, labels)

for an in results["analogies"]:
    answers = ", ".join(f"{w} ({s:.2f})" for w, s in an["answers"])
    print(f"{an['a']} : {an['b']} :: {an['c']} : ?   ->  {answers}")

The first two work in a loose sense (a negative feeling maps to a positive feeling; a language maps to another language). The third (`deadlock : thread :: off_by_one : ?`) is weak: the model mostly returns other bug-related words instead of a specific "detail" word like `range` or `index`. With a small, template-based corpus, analogies only work when the relationship is strongly represented in the data.

### 3.5 Which word doesn't belong?
`doesnt_match` picks the word furthest from the average of the list.

In [ ]:
for group in [["python", "java", "javascript", "anxiety"],
              ["copilot", "chatbot", "llm", "deadlock"],
              ["confidence", "pride", "relief", "panic"]]:
    print(group, "->", wv.doesnt_match(group))

## 4. 2D graphs

We plot the **81 topic words** (all words from `GROUPS`), colored by group. The vectors have 100 dimensions, so we need to project them down:

- **PCA** finds the 2 directions along which the vectors vary the most and projects onto them. It is linear and keeps global layout, but only a small share of the total variance fits into 2 dimensions.
- **t-SNE** moves points so that words that are close in 100D stay close in 2D. It shows local clusters well, but distances *between* clusters are not meaningful.

In [ ]:
def scatter_2d(points, title, xlabel, ylabel):
    fig, ax = plt.subplots(figsize=(12, 8))
    for group, color in COLORS.items():
        idx = [i for i, l in enumerate(labels) if l == group]
        ax.scatter(points[idx, 0], points[idx, 1], s=60, color=color,
                   edgecolor="white", linewidth=1, label=group, zorder=3)
        for i in idx:
            ax.annotate(pretty(words[i]), points[i], xytext=(4, 3),
                        textcoords="offset points", fontsize=8, color="#1E293B")
    ax.set_title(title, loc="left", fontsize=14, fontweight="bold")
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.grid(True, color="#E2E8F0", linewidth=0.6); ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, title="Topic group")
    plt.tight_layout()
    plt.show()

print(f"{len(words)} topic words plotted")
pca2 = PCA(n_components=2, random_state=SEED)
p2 = pca2.fit_transform(vectors)
ev2 = pca2.explained_variance_ratio_
scatter_2d(p2, "Word2Vec embeddings in 2D (PCA)",
           f"PC1 ({ev2[0]:.0%} of variance)", f"PC2 ({ev2[1]:.0%} of variance)")

In the PCA view the groups already form visible regions, but they overlap because two directions capture only about a fifth of the variation in 100 dimensions.

In [ ]:
tsne = TSNE(n_components=2, perplexity=12, metric="cosine", init="pca",
            random_state=SEED, max_iter=2000)
t2 = tsne.fit_transform(vectors)
scatter_2d(t2, "Word2Vec embeddings in 2D (t-SNE)", "t-SNE dimension 1", "t-SNE dimension 2")

t-SNE separates the nine topic groups into distinct islands: every word sits with the other words from its group.

## 5. 3D graphs

Adding a third PCA direction shows a bit more of the structure. First a static matplotlib view (which renders anywhere, including on GitHub):

In [ ]:
pca3 = PCA(n_components=3, random_state=SEED)
p3 = pca3.fit_transform(vectors)
ev3 = pca3.explained_variance_ratio_
print("explained variance of PC1-PC3:", np.round(ev3, 3), "total:", round(float(ev3.sum()), 3))

fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection="3d")
for group, color in COLORS.items():
    idx = [i for i, l in enumerate(labels) if l == group]
    ax.scatter(p3[idx, 0], p3[idx, 1], p3[idx, 2], s=45, color=color,
               edgecolor="white", linewidth=0.6, label=group)
    for i in idx:
        ax.text(p3[i, 0], p3[i, 1], p3[i, 2], " " + pretty(words[i]), fontsize=7)
ax.set_xlabel(f"PC1 ({ev3[0]:.0%})"); ax.set_ylabel(f"PC2 ({ev3[1]:.0%})"); ax.set_zlabel(f"PC3 ({ev3[2]:.0%})")
ax.view_init(elev=22, azim=-58)
ax.set_title("Word2Vec embeddings in 3D (PCA)", loc="left", fontsize=14, fontweight="bold")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 0.95), frameon=False)
plt.tight_layout()
plt.show()

And an **interactive** version with plotly (drag to rotate, scroll to zoom, hover for the word).

> **Note:** GitHub's notebook viewer does not run JavaScript, so the interactive figure below may appear blank there. Run the notebook locally (Jupyter / VS Code) to interact with it, or open **`outputs/embedding_3d.html`** (generated by `train_word2vec.py`) in a browser for the same interactive plot.

In [ ]:
fig = go.Figure()
for group, color in COLORS.items():
    idx = [i for i, l in enumerate(labels) if l == group]
    fig.add_trace(go.Scatter3d(
        x=p3[idx, 0], y=p3[idx, 1], z=p3[idx, 2], mode="markers+text", name=group,
        text=[pretty(words[i]) for i in idx], textposition="top center",
        textfont=dict(size=10), marker=dict(size=5, color=color),
        hovertemplate="<b>%{text}</b><br>" + group + "<extra></extra>"))
fig.update_layout(
    title="Linewise Word2Vec: interactive 3D view (drag to rotate)",
    scene=dict(xaxis_title=f"PC1 ({ev3[0]:.0%})", yaxis_title=f"PC2 ({ev3[1]:.0%})",
               zaxis_title=f"PC3 ({ev3[2]:.0%})"),
    legend_title="Topic group", height=650, margin=dict(l=0, r=0, t=50, b=0))
fig.show()

## 6. Cluster quality

The graphs are projections, so we also measure the clusters in the full 100-dimension space.

- **Silhouette score** (cosine distance): for each word, compares how close it is to its own group versus the nearest other group. Ranges from −1 to 1; above about 0.25 indicates real structure, above 0.5 is strong.
- **Within- vs across-group similarity**: the average cosine similarity between two words of the same group versus two words of different groups.

In [ ]:
sil = silhouette_score(vectors, labels, metric="cosine")

unit = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
sims = unit @ unit.T
lab = np.array(labels)
same_group = lab[:, None] == lab[None, :]
off_diag = ~np.eye(len(lab), dtype=bool)
within = sims[same_group & off_diag].mean()
across = sims[~same_group].mean()

print(f"silhouette score (cosine)     : {sil:.3f}")
print(f"mean within-group similarity  : {within:.3f}")
print(f"mean across-group similarity  : {across:.3f}")
print(f"gap                           : {within - across:.3f}")

In [ ]:
for group in GROUPS:
    m = lab == group
    w = sims[np.ix_(m, m)][~np.eye(m.sum(), dtype=bool)].mean()
    a = sims[np.ix_(m, ~m)].mean()
    print(f"{group:20s} within {w:.2f}   across {a:.2f}")

**Interpretation:** the silhouette score is clearly positive, meaning words sit closer to their own topic group than to any other group, and words within a group are on average far more similar to each other than to words in other groups. Keep in mind that the templates were written per topic, so this mostly confirms that Word2Vec picked up the context structure we built into the corpus.

## 7. Conclusion

**What the model learned.**
- Words used in the same contexts got nearby vectors: bug types cluster together, as do languages, AI tools, review verbs, practice words, career words, and negative and positive feelings.
- Nearest neighbors and `doesnt_match` behave sensibly (`anxiety` is in the "wrong" list next to programming languages; `deadlock` stands out among AI tools).
- Some relations cross topics in a meaningful way: the feeling analogy maps a negative feeling to a positive one, mirroring the Linewise story of practice turning anxiety into confidence.

**Limitations.**
- **Small corpus:** about 60k tokens; real Word2Vec models are trained on hundreds of millions to billions of words. Rare words from the course text have noisy vectors.
- **Synthetic sentences:** most of the training data comes from topic templates, so the clean clusters and high silhouette score partly reflect how the data was built, not language as people actually use it.
- **Weak third analogy:** `deadlock : thread :: off_by_one : ?` does not return a clear answer; fine-grained relations need far more varied text than templates provide.
- 2D and 3D projections only keep a small share of the variance, so the pictures are a guide, not the full story.

**Connection to Linewise.** Linewise is about reading code closely and naming what is wrong with it. An embedding like this one could support that: grouping practice snippets by bug type, finding a related bug to show next, or matching a student's free-text explanation ("the loop goes one past the end") to the right concept. With real student explanations and code review comments as training data instead of templates, the same pipeline would learn the vocabulary students actually use.